# 02 — Matrices et prétraitements (tâches 11 à 14)


## 1. Initialisation, protocole et données QC

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import experiment_config as cfg
from src.io.database_h5 import load_nir_uco_h5
from src.matrices.matrix_registry import build_matrix_output
from src.protocol_governance import sha256_file, verify_frozen_protocol
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import save_parquet
from src.workflows.matrix_preprocessing import (
    assert_wavelength_lock,
    build_matrix_coverage_table,
    build_wavelength_config,
    evaluate_balanced_sampling_grid,
    evaluate_preprocessing_grid,
    summarize_matrix_output,
    wavelength_axis_id,
)
from src.workflows.protocol_split import eligible_object_ids

pd.set_option("display.max_columns", 60)

RESULTS_TAG = (
    f"{int(cfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(cfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if cfg.USE_WAVELENGTH_WINDOW
    else cfg.DEFAULT_RESULTS_TAG
)

PROTOCOL_DIR = PROJECT_ROOT.joinpath(*cfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
QC_DIR = PROJECT_ROOT.joinpath(*cfg.QC_RESULTS_RELATIVE_DIR)
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / f"{cfg.MATRIX_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = {
    key: RESULTS_DIR / filename
    for key, filename in cfg.MATRIX_OUTPUT_FILENAMES.items()
}

protocol_verification = verify_frozen_protocol(
    PROTOCOL_DIR,
    strict=True,
)
display(protocol_verification)

split_manifest = pd.read_parquet(
    QC_DIR / cfg.QC_OUTPUT_FILENAMES["split_manifest"]
)
pixel_spectral_qc = pd.read_parquet(
    QC_DIR / cfg.QC_OUTPUT_FILENAMES["pixel_spectral_qc"]
)

object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=False,
)

if cfg.USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = (
        select_wavelength_range_from_database(
            object_db=object_db,
            image_db=image_db,
            min_nm=cfg.WAVELENGTH_WINDOW_MIN_NM,
            max_nm=cfg.WAVELENGTH_WINDOW_MAX_NM,
        )
    )
else:
    wavelengths = np.asarray(
        next(iter(object_db.values()))["wavelengths"],
        dtype=float,
    )

calibration_ids = eligible_object_ids(
    split_manifest,
    "calibration",
)
validation_ids = eligible_object_ids(
    split_manifest,
    "validation",
)

if not calibration_ids:
    raise RuntimeError(
        "No QC-eligible calibration object is available."
    )
if not validation_ids:
    raise RuntimeError(
        "No QC-eligible validation object is available."
    )

observed_calibration_batches = set(
    pd.to_numeric(
        split_manifest.loc[
            split_manifest["protocol_role"].eq("calibration")
            & split_manifest["qc_eligibility"].eq("accepted"),
            "batch",
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)

observed_validation_batches = set(
    pd.to_numeric(
        split_manifest.loc[
            split_manifest["protocol_role"].eq("validation")
            & split_manifest["qc_eligibility"].eq("accepted"),
            "batch",
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
)

if observed_calibration_batches != set(cfg.PROTOCOL_CALIBRATION_BATCHES):
    raise RuntimeError(
        "Unexpected calibration batches: "
        f"{sorted(observed_calibration_batches)}"
    )
if observed_validation_batches != set(cfg.PROTOCOL_VALIDATION_BATCHES):
    raise RuntimeError(
        "Unexpected validation batches: "
        f"{sorted(observed_validation_batches)}"
    )

print("RESULTS_TAG:", RESULTS_TAG)
print("Calibration objects:", len(calibration_ids))
print("Validation objects:", len(validation_ids))
print("Wavelength bands:", len(wavelengths))

,check,passed,detail
0,all_frozen_artifacts_exist,True,missing=[]
1,configuration_sha256_matches_current_protocol,True,expected=9e87eac5b065a9fae9ef1ff543981234bfbda...
2,inference_plan_sha256_matches_current_protocol,True,expected=10624351f77a2b18a37b8a51b766be759e4cc...
3,planned_contrasts_sha256_matches_current_protocol,True,expected=5a557c1c13441366f7795cbed2504f70516af...
4,checks_file_checksum_matches_lock,True,expected=9e972be2cfa8cfe634a304b706c12b9cab103...
5,inference_plan_file_checksum_matches_lock,True,expected=3b34fa6331ce14de3bb2b63202cbb821333cc...
6,manifest_file_checksum_matches_lock,True,expected=80fd7411e9a0493d37f60ec59c229702ee3b4...
7,planned_contrasts_file_checksum_matches_lock,True,expected=14abc9529b2fa764bc9412c0122e5c981c55a...
8,lock_checksum_is_valid,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...


RESULTS_TAG: px_qc_v1
Calibration objects: 209
Validation objects: 108
Wavelength bands: 61


## 2. Vérouillage de l'axe spectral

In [2]:
wavelength_candidate = build_wavelength_config(
    image_db,
    object_db,
    wavelength_mode=cfg.DEFAULT_WAVELENGTH_MODE,
    protocol_version=cfg.PROTOCOL_VERSION,
    n_remove_start=cfg.N_REMOVE_START,
    n_stop_end=cfg.N_STOP_END,
    window_min_nm=(
        cfg.WAVELENGTH_WINDOW_MIN_NM if cfg.USE_WAVELENGTH_WINDOW else None
    ),
    window_max_nm=(
        cfg.WAVELENGTH_WINDOW_MAX_NM if cfg.USE_WAVELENGTH_WINDOW else None
    ),
)
if wavelength_candidate.iloc[0]['wavelength_axis_id'] != wavelength_axis_id(wavelengths):
    raise RuntimeError(
        "Wavelength axis ID mismatch between candidate and actual wavelengths."
    )
if OUTPUT["wavelength_config"].exists():
    assert_wavelength_lock(
        pd.read_parquet(OUTPUT["wavelength_config"]),
        wavelength_candidate,
    )
save_parquet(wavelength_candidate, OUTPUT["wavelength_config"], optimize=False)

# Image cubes are no longer needed after the axis and band counts have
# been locked. Matrix construction below uses object spectra only.
for image in image_db.values():
    image.pop("cube", None)
    image.pop("image_ref_selected_range", None)

import gc
gc.collect()

wavelength_candidate

,protocol_version,spectral_config_id,wavelength_mode,wavelength_axis_id,n_remove_start,n_stop_end,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm,n_images_checked,n_objects_checked,all_axes_match,strictly_increasing,unique_axis,locked
0,8tracks_v5,4b157f0c2cf069cb5d88fa1106e2379b10023825553723...,non_noisy_all,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,6,67,None,None,61,960.735294,1678.088235,48,1262,True,True,True,True


## 3. Construction des matrices de calibration et de validation

In [3]:
matrix_summaries, coverage_tables, matrix_errors = [], [], []
matrix_outputs = {}
for role, ids in (("calibration", calibration_ids), ("validation", validation_ids)):
    for method, strategy, m in cfg.PREPROCESSING_MATRIX_SPECS:
        matrix_core_id = method if strategy is None else f"{method}_{strategy}_m{int(m)}"
        matrix_id = f"{role}_{matrix_core_id}"
        try:
            output = build_matrix_output(
                object_db,
                matrix_method=method,
                filters={"object_id": ids},
                m=cfg.M_BALANCED_PIXELS if m is None else int(m),
                random_state=cfg.RANDOM_STATE,
                replace=cfg.REPLACE_BALANCED_PIXELS,
                balanced_pixel_strategy="random" if strategy is None else strategy,
                under_m_policy=cfg.BALANCED_SAMPLING_UNDER_M_POLICY,
                require_two_classes=True,
                pixel_validity_policy=cfg.SPECTRAL_PIXEL_VALIDITY_POLICY,
            )
            row, _ = summarize_matrix_output(
                output.X,
                output.y,
                output.metadata,
                matrix_method=method,
                balanced_pixel_strategy=strategy,
                matrix_id=matrix_id,
                protocol_role=role,
                wavelengths=output.wavelengths,
            )
            matrix_summaries.append(row)
            coverage_tables.append(
                build_matrix_coverage_table(output.metadata, matrix_id=matrix_id)
            )
            matrix_outputs[(role, method, strategy, m)] = output
        except Exception as exc:
            matrix_errors.append(
                {
                    "matrix_id": matrix_id,
                    "protocol_role": role,
                    "matrix_method": method,
                    "balanced_pixel_strategy": strategy,
                    "m": m,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )
matrix_summary = pd.DataFrame(
    matrix_summaries, columns=cfg.MATRIX_SUMMARY_REQUIRED_COLUMNS
)
matrix_coverage = (
    pd.concat(coverage_tables, ignore_index=True)
    if coverage_tables else pd.DataFrame(columns=cfg.MATRIX_COVERAGE_COLUMNS)
)
matrix_error_table = pd.DataFrame(
    matrix_errors, columns=cfg.MATRIX_ERROR_COLUMNS
)
save_parquet(matrix_summary, OUTPUT["matrix_summary"])
save_parquet(matrix_coverage, OUTPUT["matrix_coverage"])
save_parquet(matrix_error_table, OUTPUT["matrix_errors"])

display(matrix_summary)

if matrix_errors:
    raise RuntimeError(f"Candidate matrix failures: {matrix_errors}")

,matrix_id,protocol_role,matrix_method,balanced_pixel_strategy,n_observations,n_features,n_classes,n_objects,n_images,n_nan,n_inf,matrix_rank,rank_ratio,n_zero_variance_bands,wavelength_axis_id,status
0,calibration_object_mean,calibration,object_mean,NaN,209,61,2,209,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
1,calibration_object_median,calibration,object_median,NaN,209,61,2,209,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
2,calibration_all_pixels,calibration,all_pixels,NaN,14965,61,2,209,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
3,calibration_balanced_pixels_random_m10,calibration,balanced_pixels,random,2090,61,2,209,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
4,calibration_balanced_pixels_center_m10,calibration,balanced_pixels,center,2090,61,2,209,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
5,calibration_balanced_pixels_random_m20,calibration,balanced_pixels,random,4140,61,2,207,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
6,calibration_balanced_pixels_center_m20,calibration,balanced_pixels,center,4140,61,2,207,4,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
7,validation_object_mean,validation,object_mean,NaN,108,61,2,108,2,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
8,validation_object_median,validation,object_median,NaN,108,61,2,108,2,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted
9,validation_all_pixels,validation,all_pixels,NaN,6807,61,2,108,2,0,0,61,1.0,0,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,accepted


## 4. Domaine physique des matrices

In [4]:
physical_domain_rows = []

for matrix_id, output in matrix_outputs.items():
    role, method, strategy, m = matrix_id
    X = np.asarray(output.X, dtype=float)
    finite = bool(np.isfinite(X).all())
    n_nonpositive = int(np.count_nonzero(X <= 0.0))
    physical_domain_rows.append(
        {
            "role": role,
            "matrix_method": method,
            "strategy": strategy,
            "m": m,
            "n_rows": int(X.shape[0]),
            "n_bands": int(X.shape[1]),
            "finite": finite,
            "n_nonpositive": n_nonpositive,
        }
    )
    if not np.isfinite(X).all():
        raise RuntimeError(
            f"{matrix_id}: non-finite values remain after pixel QC."
        )
    if n_nonpositive:
        raise RuntimeError(
            f"{matrix_id}: {n_nonpositive} non-positive reflectance "
            "values remain after spectral pixel filtering."
        )

physical_domain_audit = pd.DataFrame(physical_domain_rows)
physical_domain_audit

,role,matrix_method,strategy,m,n_rows,n_bands,finite,n_nonpositive
0,calibration,object_mean,NaN,NaN,209,61,True,0
1,calibration,object_median,NaN,NaN,209,61,True,0
2,calibration,all_pixels,NaN,NaN,14965,61,True,0
3,calibration,balanced_pixels,random,10.0,2090,61,True,0
4,calibration,balanced_pixels,center,10.0,2090,61,True,0
5,calibration,balanced_pixels,random,20.0,4140,61,True,0
6,calibration,balanced_pixels,center,20.0,4140,61,True,0
7,validation,object_mean,NaN,NaN,108,61,True,0
8,validation,object_median,NaN,NaN,108,61,True,0
9,validation,all_pixels,NaN,NaN,6807,61,True,0


## 5. Faisabilité du sampling équilibré (calibration only)

In [5]:
m_feasibility, pixel_sampling_diagnostics = evaluate_balanced_sampling_grid(
    object_db,
    filters={"object_id": calibration_ids},
    m_values=cfg.BALANCED_SAMPLING_M_VALUES,
    strategies=cfg.BALANCED_PIXEL_STRATEGIES,
    seeds=cfg.BALANCED_SAMPLING_SEEDS,
    replace=cfg.REPLACE_BALANCED_PIXELS,
    under_m_policy=cfg.BALANCED_SAMPLING_UNDER_M_POLICY,
    min_eligible_rate=cfg.BALANCED_SAMPLING_MIN_ELIGIBLE_RATE,
    return_diagnostics=True,
    pixel_validity_policy=cfg.SPECTRAL_PIXEL_VALIDITY_POLICY,
)
save_parquet(m_feasibility, OUTPUT["m_feasibility"])
save_parquet(pixel_sampling_diagnostics, OUTPUT["pixel_sampling_diagnostics"])
m_feasibility

,m,strategy,under_m_policy,n_objects_total,n_objects_under_m,eligible_rate,n_rows,n_classes,n_images,class_balance_ratio,image_balance_ratio,selection_stability,status
0,5,random,exclude,209,0,1.000000,1045,2,4,0.882883,0.779661,0.042872,accepted
1,5,center,exclude,209,0,1.000000,1045,2,4,0.882883,0.779661,1.000000,accepted
2,10,random,exclude,209,0,1.000000,2090,2,4,0.882883,0.779661,0.097347,accepted
3,10,center,exclude,209,0,1.000000,2090,2,4,0.882883,0.779661,1.000000,accepted
4,20,random,exclude,209,2,0.990431,4140,2,4,0.864865,0.762712,0.201115,accepted
5,20,center,exclude,209,2,0.990431,4140,2,4,0.864865,0.762712,1.000000,accepted
6,30,random,exclude,209,5,0.976077,6120,2,4,0.854545,0.745763,0.320673,accepted
7,30,center,exclude,209,5,0.976077,6120,2,4,0.854545,0.745763,1.000000,accepted
8,40,random,exclude,209,20,0.904306,7560,2,4,0.890000,0.800000,0.430790,accepted
9,40,center,exclude,209,20,0.904306,7560,2,4,0.890000,0.800000,1.000000,accepted


## 6. Validation technique des preprocessings (calibration only)

In [6]:
preprocessing_tables, preprocessing_errors = [], []

for method, strategy, m in cfg.PREPROCESSING_MATRIX_SPECS:
    fit = matrix_outputs[("calibration", method, strategy, m)]
    pair_id = method if strategy is None else f"{method}_{strategy}_m{int(m)}"
    summary, _, errors = evaluate_preprocessing_grid(
        fit.X,
        X_eval=None,
        preprocessing_configs=cfg.PREPROCESSING_CONFIGS_TO_COMPARE,
        sg_windows=cfg.SG_WINDOW_CHOICES,
        sg_polyorder=cfg.SG_POLYORDER,
        wavelengths=fit.wavelengths,
        matrix_id=pair_id,
        fit_role="calibration",
        eval_role="calibration",
    )
    preprocessing_tables.append(summary)
    if not errors.empty:
        preprocessing_errors.append(errors)

preprocessing_validation = pd.concat(preprocessing_tables, ignore_index=True)
preprocessing_error_table = (
    pd.concat(preprocessing_errors, ignore_index=True)
    if preprocessing_errors
    else pd.DataFrame(
        columns=cfg.PREPROCESSING_ERROR_COLUMNS
    )
)
if not preprocessing_validation["fit_role"].eq("calibration").all():
    raise RuntimeError("Preprocessing eligibility contains a non-calibration fit role.")

if not preprocessing_validation["eval_role"].eq("calibration").all():
    raise RuntimeError(
        "Preprocessing eligibility must be calibration-only. "
        "Batch 3 must not determine notebook-03 candidates."
    )

save_parquet(preprocessing_validation, OUTPUT["preprocessing_validation"])
save_parquet(preprocessing_error_table, OUTPUT["preprocessing_errors"])
if not preprocessing_error_table.empty:
    print(
        "Some preprocessing candidates failed technical validation; "
        "inspect preprocessing_errors.parquet."
    )
preprocessing_validation.query("status == 'accepted'")


Some preprocessing candidates failed technical validation; inspect preprocessing_errors.parquet.


,matrix_id,fit_role,eval_role,wavelength_axis_id,preprocessing,steps,sg_window_length,sg_polyorder,deriv,status,n_features_before,n_features_after,band_count_unchanged,n_nan,n_inf,zero_variance_band_rate,saturation_rate,global_min,global_max,repeatability_error,name_steps_coherent
0,object_mean,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,raw,raw,NaN,NaN,NaN,accepted,61,61,True,0,0,0.0,0.0,0.135811,0.599930,0.0,True
1,object_mean,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,absorbance,absorbance,NaN,NaN,NaN,accepted,61,61,True,0,0,0.0,0.0,0.221899,0.867065,0.0,True
2,object_mean,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,snv,snv,NaN,NaN,NaN,accepted,61,61,True,0,0,0.0,0.0,-1.631037,1.277540,0.0,True
3,object_mean,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,msc,msc,NaN,NaN,NaN,accepted,61,61,True,0,0,0.0,0.0,0.217821,0.455964,0.0,True
4,object_mean,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,vector_norm,vector_norm,NaN,NaN,NaN,accepted,61,61,True,0,0,0.0,0.0,0.067812,0.172570,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,balanced_pixels_center_m20,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,absorbance_snv_sg_d2,absorbance + snv + sg_d2,7.0,2.0,2.0,accepted,61,61,True,0,0,0.0,0.0,-0.002187,0.002223,0.0,True
549,balanced_pixels_center_m20,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,absorbance_snv_sg_d2,absorbance + snv + sg_d2,9.0,2.0,2.0,accepted,61,61,True,0,0,0.0,0.0,-0.001610,0.001133,0.0,True
550,balanced_pixels_center_m20,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,absorbance_snv_sg_d2,absorbance + snv + sg_d2,11.0,2.0,2.0,accepted,61,61,True,0,0,0.0,0.0,-0.001186,0.000701,0.0,True
551,balanced_pixels_center_m20,calibration,calibration,02c5718c409fa484b1b90176a4a1610335c149c0aa6554...,absorbance_snv_sg_d2,absorbance + snv + sg_d2,13.0,2.0,2.0,accepted,61,61,True,0,0,0.0,0.0,-0.000864,0.000584,0.0,True


## 7. Audit de l'exclusion effective des pixels QC invalides

In [7]:
invalid_pixel_qc = (
    pixel_spectral_qc.loc[
        ~pixel_spectral_qc["analysis_valid"],
        [
            "object_id",
            "pixel_index",
            "row",
            "col",
            "invalid_reason",
        ],
    ]
    .copy()
)
display(invalid_pixel_qc)

invalid_keys = set(
    zip(
        invalid_pixel_qc["object_id"].astype(str),
        invalid_pixel_qc["pixel_index"].astype(int),
    )
)

audit_rows = []
for key, output in matrix_outputs.items():
    role, method, strategy, m = key
    X = np.asarray(output.X,dtype=float)
    overlap_count = 0

    if output.matrix_spec.level in {"pixel", "balanced_pixel"}:
        meta = pd.DataFrame(output.metadata)
        observed_keys = set(
            zip(
                meta["object_id"].astype(str),
                meta["pixel_index"].astype(int),
            )
        )
        overlap = invalid_keys & observed_keys
        overlap_count = len(overlap)
        if overlap:
            raise RuntimeError(
                f"{key}: invalid QC pixels entered the matrix: {sorted(overlap)[:10]}"
            )

    audit_rows.append(
        {
            "role": role,
            "matrix_method": method,
            "strategy": strategy,
            "m": m,
            "n_rows": int(X.shape[0]),
            "n_bands": int(X.shape[1]),
            "n_invalid_pixel_overlap": overlap_count,
        }
    )

matrix_pixel_validity_audit = pd.DataFrame(audit_rows)

matrix_pixel_validity_audit

,object_id,pixel_index,row,col,invalid_reason
654,alm1pea1_obj008,37,120,137,all_zero_spectrum
1184,alm1pea1_obj015,88,150,89,all_zero_spectrum
1913,alm1pea1_obj023,119,194,70,all_zero_spectrum
2484,alm1pea1_obj029,4,228,200,all_zero_spectrum
3367,alm1pea1_obj038,54,278,52,all_zero_spectrum
...,...,...,...,...,...
98368,peanut3_obj010,7,119,237,all_zero_spectrum
98453,peanut3_obj011,13,122,61,all_zero_spectrum
98950,peanut3_obj019,10,163,142,all_zero_spectrum
99359,peanut3_obj026,4,189,127,all_zero_spectrum


,role,matrix_method,strategy,m,n_rows,n_bands,n_invalid_pixel_overlap
0,calibration,object_mean,NaN,NaN,209,61,0
1,calibration,object_median,NaN,NaN,209,61,0
2,calibration,all_pixels,NaN,NaN,14965,61,0
3,calibration,balanced_pixels,random,10.0,2090,61,0
4,calibration,balanced_pixels,center,10.0,2090,61,0
5,calibration,balanced_pixels,random,20.0,4140,61,0
6,calibration,balanced_pixels,center,20.0,4140,61,0
7,validation,object_mean,NaN,NaN,108,61,0
8,validation,object_median,NaN,NaN,108,61,0
9,validation,all_pixels,NaN,NaN,6807,61,0


## 8. Contrôle exact de la représentation all_pixels

In [8]:
all_pixel_count_checks = []

for role, ids in (
    ("calibration", calibration_ids),
    ("validation", validation_ids),
):
    expected = (
        pixel_spectral_qc.loc[
            pixel_spectral_qc["object_id"].astype(str).isin(set(map(str, ids)))
            & pixel_spectral_qc["analysis_valid"]]
        .groupby("object_id")
        .size()
        .sort_index()
    )
    output = matrix_outputs[
        (
            role,
            "all_pixels",
            None,
            None,
        )
    ]

    observed_meta = pd.DataFrame(output.metadata)
    observed = observed_meta.groupby("object_id").size().sort_index()
    expected.index = expected.index.astype(str)
    observed.index = observed.index.astype(str)
    comparison = pd.concat({"expected_valid_pixels": expected, "observed_matrix_rows": observed}, axis=1).fillna(0)
    mismatch = comparison.nunique(axis=1) > 1
    if mismatch.any():
        raise RuntimeError(
            f"{role}: all_pixels does not match pixel-level QC:\n"
            f"{comparison.loc[mismatch].head(20)}"
        )

    all_pixel_count_checks.append(
        {
            "role": role,
            "n_objects": int(len(comparison)),
            "n_expected_rows": int(comparison["expected_valid_pixels"].sum()),
            "n_observed_rows": int(comparison["observed_matrix_rows"].sum()),
            "passed": True,
        }
    )
    
display(pd.DataFrame(all_pixel_count_checks))

,role,n_objects,n_expected_rows,n_observed_rows,passed
0,calibration,209,14965,14965,True
1,validation,108,6807,6807,True


## 9. Vérification finale des artefacts

In [9]:
required_output_keys = (
    "wavelength_config",
    "m_feasibility",
    "pixel_sampling_diagnostics",
    "matrix_summary",
    "matrix_coverage",
    "matrix_errors",
    "preprocessing_validation",
    "preprocessing_errors",
)

missing_outputs = [
    str(OUTPUT[key])
    for key in required_output_keys
    if not OUTPUT[key].exists()
]
if missing_outputs:
    raise RuntimeError(f"Notebook-02 outputs are missing: {missing_outputs}")

saved_preprocessing = pd.read_parquet(OUTPUT["preprocessing_validation"])
if not (
    saved_preprocessing["fit_role"].eq("calibration").all()
    and saved_preprocessing["eval_role"].eq("calibration").all()
):
    raise RuntimeError("Persisted preprocessing eligibility is not calibration-only.")

output_hashes_df = pd.DataFrame(
    [
        {
            "artifact": key,
            "path": str(OUTPUT[key]),
            "sha256": sha256_file(OUTPUT[key]),
        }
        for key in required_output_keys
    ]
)

display(output_hashes_df)

print("Notebook 02 complete.")
print("Results directory:", RESULTS_DIR.resolve())
print(
    "Accepted preprocessing rows:",
    int(saved_preprocessing["status"].eq("accepted").sum()),
)

,artifact,path,sha256
0,wavelength_config,C:\Users\alixg\OneDrive - Université Paris-Dau...,d7f1f4d4abb9a3b399fe9fd27027e8ffa71a0abe107e2e...
1,m_feasibility,C:\Users\alixg\OneDrive - Université Paris-Dau...,9d3c19e9b6d2cc0fd0c80021672483b5abec7af74cb997...
2,pixel_sampling_diagnostics,C:\Users\alixg\OneDrive - Université Paris-Dau...,ec01a702736d51efd46bf87e9f6c6ee890142f41fe63b7...
3,matrix_summary,C:\Users\alixg\OneDrive - Université Paris-Dau...,ab6bf26d1808438c79a7c2fa30e0b2a27ad68433b437ae...
4,matrix_coverage,C:\Users\alixg\OneDrive - Université Paris-Dau...,c641e0026fd999dcecdfb23ac5fa2889ab3e74734ed280...
5,matrix_errors,C:\Users\alixg\OneDrive - Université Paris-Dau...,b84f8e76f2c563761a7ee4006884eb0b2a3c7f4fe6317c...
6,preprocessing_validation,C:\Users\alixg\OneDrive - Université Paris-Dau...,9311cffc14706b43c716698e464d70039dfb2a595dacee...
7,preprocessing_errors,C:\Users\alixg\OneDrive - Université Paris-Dau...,38c6467a5444cab667f653024f1d8b44878ac59ff649f6...


Notebook 02 complete.
Results directory: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\02_matrices_8tracks_v5_px_qc_v1
Accepted preprocessing rows: 551
